In [0]:
from pyspark.sql.functions import coalesce, collect_set, sha2, collect_list, substring, regexp_replace, concat_ws,concat, first, col, when, rank, datediff, months_between, lit, to_timestamp, abs as pyspark_abs, DataFrame, trim,current_timestamp,from_utc_timestamp
from pyspark.sql import Window, Row
from pyspark.sql.types import StringType, BooleanType, NullType, IntegerType
from datetime import timedelta, datetime 
import boto3
from pyspark.sql.utils import AnalysisException
import os
import botocore.session
import delta_table_creator, transform
import delta  # type: ignore
from delta import DeltaTable  # type: ignore

import pyspark.sql.functions as F

# Get the current date to add to the csv filename
curr_date = datetime.now().strftime("%Y%m%d")

botosession = boto3.session.Session(region_name="eu-central-1")

# Initialize the SSM client
ssm_client = botosession.client("ssm", region_name="eu-central-1")

response = ssm_client.get_parameter(
    Name='/cfn/bucket_prefix',
    WithDecryption=False
)

env_prefix = response['Parameter']['Value']

# Initialize the S3 client
s3_client = botosession.client("s3", region_name="eu-central-1")

def has_column(df, col):
    try:
        df[col]
        return True
    except AnalysisException:
        return False

In [0]:
result = spark.sql( f'''
    SELECT max(BusinessDate) max_date
        FROM `{env_prefix}-cdh`.primary.Auction 
        WHERE SourceSystem = 'extfile'
        '''
)
max_date = result.collect()[0]['max_date']
max_date = max_date.strftime('%Y-%m-%d')


In [0]:
#run_date = dbutils.widgets.get("run_date")
run_date = max_date
run_date2 = datetime.strptime(run_date, '%Y-%m-%d') - timedelta(days=1)
run_date2 = run_date2.strftime('%Y-%m-%d')

In [0]:
schema = (
    "AuctionID  string"
    ",SourceSystem string"
    ",IngestionTimeStamp timestamp"
    ",BatchId string"
    ",AuctionKey    string"
    ",RelatedAuctionKey  string"
    ",OriginAuctionKey string"
    ",ConductionDate    date"
    ",PublishedDate date"
    ",FirstOfferPrice   decimal(18,2)"
    ",HastenerCode  string"
    ",OriginHastener  string"
    ",Hastener string"
    ",IsHastenerServicer string"
    ",Initiator string"
    ",InitiatorCepalAB string"
    ",PortfolioCategory string"
    ",HastenerTaxRegistrationNumber   string"
    ",HastenerCountry   string"
    ",AuctionStatusCode string"
    ",AuctionStatus string"
    ",DebtorPartyKey string"
    ",DebtorTaxRegistrationNumber   string"
    ",DebtorDescription    string"
    ",CourtType string"
    ",AuctionType   string"
    ",MultipleDeptorsFlag   boolean"
    ",Servicer  string"
    ",HasJudgmentFlag   boolean"
    ",FileDate   date"
    ",FileName   string"
    ",ReferenceDate date"
)

In [0]:
partition_columns = ["ReferenceDate"]


derivative_table_path = f"`{env_prefix}-cdh`.`derivativeactive`.`DAuction`"

# ================== Create Delta Table ==================
    
# logger.info("Creating Delta Table...")

if spark.catalog.tableExists(derivative_table_path):
    delta_table_creator.create_unity_delta_table(
    spark, derivative_table_path, partition_columns, schema # type: ignore
)

# ================== Delete From Delta Table ==================
            
# we have to investigate the transform function since we dont have source_system ( None )
if spark.catalog.tableExists(derivative_table_path):
    spark.sql(f"DELETE FROM {derivative_table_path} WHERE ReferenceDate = {run_date!r}")			

In [0]:
Auction_df = spark.sql(f'''
          SELECT * 
          FROM `{env_prefix}-cdh`.primary.Auction
          WHERE SourceSystem = 'extfile'
          AND BusinessDate = {run_date!r}
          ''')

AuctionRealEstate_df = spark.sql(f'''
          SELECT * 
          FROM `{env_prefix}-cdh`.derivative.DAuctionRealEstate
          WHERE SourceSystem = 'extfile'
          AND BusinessDate = {run_date2!r}
          ''')

newsphone_basic_data_df = spark.sql(f'''
          SELECT * 
          FROM `{env_prefix}-cdh`.landed.newsphone_basic_data
          WHERE BusinessDate = {run_date!r}
          ''')

In [0]:
AuctionRealEstate_extended = AuctionRealEstate_df.withColumn(
                "AuctionIdNew",
                sha2(
                    concat_ws(";",AuctionRealEstate_df.AuctionKey, AuctionRealEstate_df.SourceSystem),
                    256,
                ),
            ).withColumn("IngestionTimestampNew",
                lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
                )
            )
newsphone_basic_data_df.createOrReplaceTempView("newsphone_basic_data")
Auction_extended = Auction_df.withColumn("IngestionTimestampNew",
                lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
                )
            )

Auction_extended.createOrReplaceTempView("Auction_extended")
AuctionRealEstate_extended.createOrReplaceTempView("AuctionRealEstate_extended")


In [0]:
Auction_transform = spark.sql("""
    SELECT 
     a.AuctionID  
    ,a.SourceSystem 
    ,a.IngestionTimeStampNew as IngestionTimeStamp
    ,a.BatchId 
    ,a.AuctionKey
    ,a.RelatedAuctionKey  
    ,a.OriginAuctionKey 
    ,a.ConductionDate   
    ,a.PublishedDate
    ,a.FirstOfferPrice   
    ,a.HastenerCode 
    ,a.Hastener as OriginHastener
    ,a.HastenerTaxRegistrationNumber   
    ,a.HastenerCountry  
    ,a.AuctionStatusCode 
    ,a.AuctionStatus 
    ,a.DebtorPartyKey 
    ,a.DebtorTaxRegistrationNumber   
    ,a.DebtorDescription 
    ,a.CourtType
    ,a.AuctionType   
    ,a.MultipleDeptorsFlag 
    ,a.Servicer  
    ,a.HasJudgmentFlag   
    ,nbd.file_date as FileDate  
    ,nbd.file_name as FileName  
    ,a.BusinessDate as ReferenceDate

    FROM Auction_extended a
    INNER JOIN newsphone_basic_data nbd
        ON a.AuctionKey = nbd.UniqueCode
""")


In [0]:
AuctionRealEstate_extended_missing = spark.sql('''
    SELECT are.*
    FROM AuctionRealEstate_extended are
    LEFT JOIN Auction_extended a
        ON a.AuctionKey = are.AuctionKey
    WHERE a.AuctionKey IS NULL
'''
)
AuctionRealEstate_extended_missing.createOrReplaceTempView("AuctionRealEstate_extended_missing")


In [0]:
AuctionRealEstate_transform = spark.sql(f"""
    SELECT AuctionID
          ,SourceSystem
          ,IngestionTimeStamp
          ,BatchId
          ,AuctionKey
          ,RelatedAuctionKey
          ,OriginAuctionKey
          ,ConductionDate
          ,PublishedDate
          ,FirstOfferPrice
          ,HastenerCode
          ,Hastener as OriginHastener
          ,HastenerTaxRegistrationNumber
          ,HastenerCountry
          ,AuctionStatusCode
          ,AuctionStatus
          ,DebtorPartyKey
          ,DebtorTaxRegistrationNumber
          ,DebtorDescription
          ,CourtType
          ,AuctionType
          ,MultipleDeptorsFlag
          ,Servicer
          ,HasJudgmentFlag
          ,FileDate
          ,FileName
          ,ReferenceDate
    FROM (
        SELECT DISTINCT
         a.AuctionIdNew as AuctionID  
        ,a.SourceSystem 
        ,a.IngestionTimestampNew as IngestionTimeStamp
        ,NULL as BatchId 
        ,a.AuctionKey
        ,NULL as RelatedAuctionKey  
        ,a.AuctionKey AS OriginAuctionKey 
        ,a.AuctionDate as ConductionDate
        ,a.AuctionPublishedDate AS PublishedDate
        ,NULL as FirstOfferPrice   
        ,NULL HastenerCode 
        ,Hastener as Hastener
        ,a.HastenerTaxRegistrationNumber
        ,NULL AS HastenerCountry  
        ,NULL AS AuctionStatusCode 
        ,NULL AS AuctionStatus 
        ,NULL AS DebtorPartyKey 
        ,OriginDebtorTaxRegistrationNumber AS DebtorTaxRegistrationNumber   
        ,OriginDebtorDescription as DebtorDescription 
        ,NULL as CourtType
        ,NULL as AuctionType   
        ,NULL as  MultipleDeptorsFlag 
        ,NULL as  Servicer  
        ,NULL as HasJudgmentFlag   
        ,a.FileDate as FileDate  
        ,a.FileName as FileName  
        ,cast({run_date!r} as date) as ReferenceDate
        ,ROW_NUMBER() over (partition by a.AuctionKey order by AuctionCode  asc ) rn
        FROM AuctionRealEstate_extended_missing a
    )
    WHERE rn = 1
""")

In [0]:
final_df = AuctionRealEstate_transform.union(Auction_transform)
AuctionRealEstate_transform.createOrReplaceTempView("AuctionRealEstate_transform")

# Add ARE Hastener fallback for classifier (aligned with TIA report logic)
# When OriginHastener is NULL (Auction-sourced records with no hastener),
# fall back to DAuctionRealEstate.Hastener for the same AuctionKey.
# This ensures the classifier (exact-match + ai_classify) sees the same text
# that TIA and DAuction360Matching display via COALESCE(OriginHastener, are.Hastener).
are_hastener_fallback = spark.sql("""
    SELECT AuctionKey, FIRST(Hastener) AS ARE_Hastener_Fallback
    FROM AuctionRealEstate_extended
    WHERE Hastener IS NOT NULL
    GROUP BY AuctionKey
""")

final_df = final_df.join(are_hastener_fallback, on="AuctionKey", how="left")

# Overwrite OriginHastener with fallback value when NULL
# (consistent with cell 10 which already sets OriginHastener=ARE.Hastener for ARE-only records)
final_df = final_df.withColumn(
    "OriginHastener",
    coalesce(col("OriginHastener"), col("ARE_Hastener_Fallback"))
).drop("ARE_Hastener_Fallback")


In [0]:
%sql
SELECT AuctionKey 
FROM AuctionRealEstate_transform
group by AuctionKey 
having count(*)>1

In [0]:
%run ./InitiatorClassifier_LLM

In [0]:
final_df.createOrReplaceTempView("final_df")

final_df = spark.sql("""
        SELECT AuctionID
          ,SourceSystem
          ,IngestionTimeStamp
          ,BatchId
          ,AuctionKey
          ,RelatedAuctionKey
          ,OriginAuctionKey
          ,ConductionDate
          ,PublishedDate
          ,FirstOfferPrice
          ,HastenerCode
          ,OriginHastener
          ,Hastener
          ,IsHastenerServicer
          ,Initiator
          ,InitiatorCepalAB
          ,PortfolioCategory
          ,HastenerTaxRegistrationNumber
          ,HastenerCountry
          ,AuctionStatusCode
          ,AuctionStatus
          ,DebtorPartyKey
          ,DebtorTaxRegistrationNumber
          ,DebtorDescription
          ,CourtType
          ,AuctionType
          ,MultipleDeptorsFlag
          ,Servicer
          ,HasJudgmentFlag
          ,FileDate
          ,FileName
          ,ReferenceDate
          from final_df
""")

 

In [0]:
final_df.write.format("delta").partitionBy("ReferenceDate").mode("append").saveAsTable(f"`{env_prefix}-cdh`.`derivativeactive`.`DAuction`")
